# 🤖 02 · 모델 실행 및 실험 평가 — 학습 없이 Generate + 종합·질문별 RAGAS
**01_rag_build.ipynb 에서 만든 인덱스(FAISS · BM25 · 임베딩 모델)를 그대로 불러옵니다.** 파싱·임베딩·학습은 하지 않습니다(`generate_only=True`).

```
사용자가 챗봇에 친 질문(그대로) ─▶ BM25 × Dense(FAISS) 하이브리드 검색 ─▶ gpt-5-mini 답변 ─▶ RAGAS(전체 종합 → 질문별)
```
- 질문(`0-3`)은 챗봇 입력창에 적은 문장 그대로입니다. 오타·한영 자판·한자·영어 혼용도 **전처리·가공 없이** 보냅니다. 정답이나 메타데이터도 붙이지 않습니다.
- 출력: **① 전체 질문 종합 RAGAS 표 → ② 질문별 RAGAS 표** (Faithfulness · Answer Relevancy · Context Precision · **Answer Correctness**)
- Answer Correctness 는 정답이 필요해서, 정답은 질문과 따로 `0-4` 셀에 적습니다(질문 문장은 그대로).
- 조절할 수 있는 것: temperature · top_p · max tokens · top-k · Dense:BM25 비율 · fusion · RAGAS 지표·심판 모델 (`0-2`)
- ⏸ ■ = 안전 중지(질문 1건마다 저장). 같은 셀을 다시 실행하면 이어서 진행하고, 설정·질문이 같으면 저장된 답변·점수를 재사용합니다.
- **원격 GCP(g2-standard-4 + L4)에서 실행:** 01 을 VM 에서 돌렸거나, 로컬에서 만든 `artifacts` 폴더를 VM 프로젝트 폴더에 통째로 올리면 됩니다(윈도우 경로는 `0-2` 가 자동 변환). OpenAI 키는 `.env`, HuggingFace 는 `hf auth login` 정보를 씁니다(입력창 없음).
- **토큰 절약:** `0-2`에서 답변·심판 모델과 **추론 강도(none~max)**를 직접 고르고, `EVAL_MODE="free"`(골든 데이터셋, 토큰 0)로 평가할 수 있습니다. `1-3`·`2`에서 실제 사용 토큰과 예상 비용을 보여줍니다.
- ⚠️ **노트북은 한 번에 하나만 실행하세요** (VM RAM 16GB). 다른 노트북 커널이 남아 있으면 메모리 부족으로 커널이 강제 종료됩니다. `0-1` 셀이 다른 커널과 RAM·GPU 사용량을 보여주고, `CLOSE_OTHER_KERNELS = True` 로 정리할 수 있습니다.
- **🔎 검색 파이프라인(1-1b):** ① GPT 가 질문을 검색용 질문으로 해석(오타·한영 자판·한자·영어 — 추론 강도 low~high 조절) → ② 원래 질문 + 다시 쓴 질문으로 각각 BM25+Dense 검색 후 RRF 병합 → ③ 리랭커(내 GPU) → ④ GPT 답변 — 원래 질문 + 검색용 질문(GPT 해석)을 함께 받아 답함(`ANSWER_WITH_REWRITE`).
- **💵 비용:** GPT 를 쓰는 셀마다 그 셀에서 쓴 토큰과 달러를 출력하고, 마지막 `4)` 셀에서 단계별 합계를 보여줍니다.

## 0. 환경 · 생성 파라미터 · 질문
> 👥 **담당:** 0-1 PM · 0-2 검색 설정 = 검색 · 리랭킹, LLM·질문 해석 설정 = 질문 해석 · 답변 · 0-3~0-5 = 평가 · 데이터셋

In [1]:
# 0-1) 경로 · 토큰(OpenAI .env / HuggingFace 로그인) · CPU/GPU 버전 확인   — 보통 10~30초
#  ▸ 원격 GCP: g2-standard-4 (vCPU 4 · RAM 16GB) + NVIDIA L4 24GB — VS Code Remote-SSH / JupyterHub 커널에서 실행
#  ▸ HuggingFace 로그인(선택, VM 터미널에서 한 번): hf auth login   ← 이후 HF_TOKEN 입력창 없음
#  ▸ ⑤ 메모리 점검: 다른 노트북 커널이 RAM·GPU 를 잡고 있으면 경고 (CLOSE_OTHER_KERNELS=True 로 종료)
import os, sys, time
from pathlib import Path

T0 = time.time()
PROJECT_DIR = Path.cwd() if (Path.cwd() / "rag").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10 이상이 필요합니다 (현재 {sys.version.split()[0]}) → 커널을 Python 3.10 으로 바꾸세요")

HF_CACHE_DIR = None        # 모델 저장 위치(기본 ~/.cache/huggingface, 약 10GB). 디스크가 부족하면 예) "/mnt/disk/hf_cache"
NEED_OPENAI  = True       # 02: 답변 생성·RAGAS 에 필수

print("① torch 로딩…", flush=True)
from rag.hw import apply_runtime_env, detect, gpu_mem, summary, require_gpu
apply_runtime_env(cache_dir=HF_CACHE_DIR)
print(f"   완료 {time.time() - T0:.1f}s", flush=True)

print("② 토큰 — OpenAI: .env / HuggingFace: 로그인 정보(hf auth login) 사용, HF_TOKEN 입력창 없음", flush=True)
from rag.credentials import load_tokens, verify_tokens, mask
import pandas as pd
tok_status = load_tokens(PROJECT_DIR / ".env", ask=False, save_to_env=False, required=())   # 입력창을 띄우지 않음
_k = os.environ.get("OPENAI_API_KEY", "").strip()
while _k.startswith("sk-sk-"):                    # 'sk-' 가 두 번 붙은 키 바로잡기 (메모리에서만)
    _k = _k[3:]
if _k:
    os.environ["OPENAI_API_KEY"] = _k
elif NEED_OPENAI:
    import getpass
    print("🔑 OPENAI_API_KEY 가 .env 에 없습니다 → 입력창(VS Code: 창 맨 위)에 붙여넣고 Enter", flush=True)
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ").strip()
    tok_status["OPENAI_API_KEY"]["출처"] = "입력(메모리에만)"

# HuggingFace: 터미널에서 한 번 'hf auth login'(구버전: huggingface-cli login) 하면 저장된 로그인 정보를 자동 사용
from huggingface_hub import get_token
_hf = get_token() or next((p.read_text().strip() for p in [Path.home() / ".cache" / "huggingface" / "token"] if p.exists()), None)
if _hf:
    os.environ["HF_TOKEN"] = _hf
oa = os.environ.get("OPENAI_API_KEY", "")
display(pd.DataFrame({
    "OPENAI_API_KEY": {"값": mask(oa), "출처": tok_status["OPENAI_API_KEY"]["출처"] if oa else "-",
                       "용도": "gpt-5-mini 생성·RAGAS 심판" + ("" if NEED_OPENAI else " (01 에서는 파인튜닝 QA 생성 때만)")},
    "HuggingFace": {"값": "로그인됨" if _hf else "로그인 안 됨", "출처": "hf auth login" if _hf else "-",
                    "용도": "모델 다운로드 (KURE-v1·Qwen2.5-VL 은 공개 모델이라 로그인 없이도 받아짐)"},
}).T)
print("③ 토큰 검증 (각 최대 15초)", flush=True)
tok_check = verify_tokens("gpt-5-mini")
print("검증:", tok_check)
if NEED_OPENAI and any(e in tok_check.get("OPENAI_API_KEY", "") for e in ("AuthenticationError", "PermissionDeniedError", "NotFoundError")):
    raise RuntimeError(f"OpenAI 키를 쓸 수 없습니다: {tok_check['OPENAI_API_KEY']} · 현재 키 {mask(os.environ.get('OPENAI_API_KEY'))}"
                       " → .env 의 OPENAI_API_KEY 확인 후 이 셀을 다시 실행하세요")

print("④ GPU 점검", flush=True)
REQUIRE_GPU = False         # 생성은 API, GPU(L4)는 질문 임베딩에만 사용 — 없으면 CPU
HW = detect()
from rag.hw import hw_table
display(hw_table(HW).rename(columns={"값": "지금 사용 중인 CPU · GPU · 버전"}))
if HW.get("gpu"):
    print(("✅ " if HW.get("kernel_ok") else "❌ ") + str(HW.get("kernel_msg")))
print(gpu_mem())
if REQUIRE_GPU:
    require_gpu(HW)
print("⑤ 메모리 점검", flush=True)
# ⑤ 메모리 점검 — VM RAM 16GB: 다른 노트북 커널이 남아 있으면 메모리가 모자라 리눅스가 커널을 강제 종료(OOM)합니다
CLOSE_OTHER_KERNELS = False   # True: 이 노트북 말고 떠 있는 다른 Jupyter 커널을 종료해 RAM·GPU 확보 (그 노트북들의 변수는 사라짐)
import psutil, subprocess
def _gb(b): return round(b / 1024**3, 1)
_me = psutil.Process()
_gpu = {}
try:
    _o = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True, timeout=10).stdout
    for _ln in _o.strip().splitlines():
        _p, _m = [x.strip() for x in _ln.split(",")]
        _gpu[int(_p)] = round(int(_m) / 1024, 1)
except Exception:
    pass
_others = []
for _p in psutil.process_iter(["pid", "cmdline", "memory_info", "create_time"]):
    try:
        _cl = _p.info["cmdline"] or []
        if (_p.info["pid"] != _me.pid and _cl and "python" in os.path.basename(_cl[0]).lower()
                and any("ipykernel" in a for a in _cl[1:])):          # 다른 노트북 커널 (python -m ipykernel_launcher …)
            _others.append(_p)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
if _others:
    display(pd.DataFrame([{"PID": p.pid, "RAM(GB)": _gb(p.info["memory_info"].rss), "GPU(GB)": _gpu.get(p.pid, 0.0),
                           "켜진 지(분)": int((time.time() - p.info["create_time"]) / 60)} for p in _others])
            .set_index("PID").rename_axis("다른 Jupyter 커널"))
    if CLOSE_OTHER_KERNELS:
        for p in _others:
            p.terminate()
        _gone, _alive = psutil.wait_procs(_others, timeout=8)
        for p in _alive:
            p.kill()
        print(f"🧹 다른 커널 {len(_others)}개 종료 → RAM·GPU 반환", flush=True)
        time.sleep(2)
    else:
        print(f"⚠️ 다른 Jupyter 커널 {len(_others)}개가 RAM {sum(_gb(p.info['memory_info'].rss) for p in _others):.1f}GB · "
              f"GPU {sum(_gpu.get(p.pid, 0) for p in _others):.1f}GB 를 쓰고 있습니다 → 메모리 부족으로 커널이 강제 종료될 수 있음\n"
              f"   끄는 법: 해당 노트북에서 커널 종료(Shut Down) · 터미널 'kill {' '.join(str(p.pid) for p in _others)}' · "
              "또는 위 CLOSE_OTHER_KERNELS = True 후 이 셀 재실행", flush=True)
_vm, _sw = psutil.virtual_memory(), psutil.swap_memory()
print(f"RAM 사용 가능 {_gb(_vm.available)} / {_gb(_vm.total)}GB · 이 커널 {_gb(_me.memory_info().rss)}GB · "
      f"스왑 {_gb(_sw.total)}GB · {gpu_mem()}", flush=True)
if _vm.available < 4 * 1024**3:
    print("⚠️ 사용 가능한 RAM 이 4GB 미만입니다 → 다른 커널을 끄거나, 노트북은 한 번에 하나만 실행하세요", flush=True)
if _sw.total == 0:
    print("ℹ️ 스왑이 없습니다. 메모리가 잠깐 모자랄 때 커널이 바로 죽지 않게 하려면 VM 터미널에서 한 번:\n"
          "   sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile", flush=True)
print(f"⏱ 총 {time.time() - T0:.1f}s")

① torch 로딩…
   완료 5.7s
② 토큰 — OpenAI: .env / HuggingFace: 로그인 정보(hf auth login) 사용, HF_TOKEN 입력창 없음
✔ .env 읽음: /home/spai1322/rfp_rag/.env  (키: OPENAI_API_KEY)


,값,출처,용도
OPENAI_API_KEY,sk-pr…6mQA,.env,gpt-5-mini 생성·RAGAS 심판
HuggingFace,로그인 안 됨,-,모델 다운로드 (KURE-v1·Qwen2.5-VL 은 공개 모델이라 로그인 없이도 ...


③ 토큰 검증 (각 최대 15초)
… OpenAI 키 확인 중
  (3.3s)
검증: {'OPENAI_API_KEY': '✅ 유효 · gpt-5-mini 접근 가능', 'HF_TOKEN': '— 미설정 (공개 모델은 토큰 없이도 받을 수 있음)'}
④ GPU 점검
… GPU 커널 점검 중 (첫 실행은 CUDA 초기화로 수 초~수십 초)


,지금 사용 중인 CPU · GPU · 버전
CPU,Intel(R) Xeon(R) CPU @ 2.20GHz
CPU 코어·스레드,물리 2코어 / 논리 4스레드
RAM,15.6 GB
GPU,NVIDIA L4
GPU 메모리,22.0 GB
GPU 아키텍처,sm_89 · bf16 지원
NVIDIA 드라이버,580.178.04
PyTorch,2.11.0+cu128
CUDA (torch 빌드),12.8
cuDNN,9.19.0


✅ CUDA 커널 실행 OK (fp16/bf16 matmul, softmax) (0.9s)
GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 11.1 / 22.0GB
⑤ 메모리 점검


,RAM(GB),GPU(GB),켜진 지(분)
다른 Jupyter 커널,,,
21121,0.0,0.0,6109
21255,0.0,0.0,6063
26656,0.0,0.0,4628
30472,2.3,7.5,4569
31252,0.0,0.0,4499
39221,2.6,2.9,3216
54510,0.6,0.2,48


⚠️ 다른 Jupyter 커널 7개가 RAM 5.5GB · GPU 10.6GB 를 쓰고 있습니다 → 메모리 부족으로 커널이 강제 종료될 수 있음
   끄는 법: 해당 노트북에서 커널 종료(Shut Down) · 터미널 'kill 21121 21255 26656 30472 31252 39221 54510' · 또는 위 CLOSE_OTHER_KERNELS = True 후 이 셀 재실행
RAM 사용 가능 3.5 / 15.6GB · 이 커널 1.2GB · 스왑 0.0GB · GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 11.1 / 22.0GB
⚠️ 사용 가능한 RAM 이 4GB 미만입니다 → 다른 커널을 끄거나, 노트북은 한 번에 하나만 실행하세요
ℹ️ 스왑이 없습니다. 메모리가 잠깐 모자랄 때 커널이 바로 죽지 않게 하려면 VM 터미널에서 한 번:
   sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile
⏱ 총 11.3s


In [2]:
# 0-2) ★★ 생성 · 검색 · 평가 파라미터 (학습 없이 바로 반영) ★★
WORK_DIR = "artifacts"                       # 01 노트북의 WORK_DIR 과 같아야 함

# ─────────────────────────────────────────── ⚖️ 하이브리드 검색 (BM25 : Dense) · top-k
TOP_K        = 5
DENSE_WEIGHT = 0.8                           # Dense 비율 (BM25 = 1 − 0.6 = 0.4)
FUSION       = "score"                       # score(min-max 가중합) | rrf
CANDIDATE_K, RRF_K = 50, 60
HNSW_EF_SEARCH = 128                         # faiss index_type="hnsw" 일 때만 (재구축 불필요)
ROUTING = dict(enabled=True, auto_extract_orgs=True, followup_query="none", orgs={})   # 질문 속 기관명 → 해당 문서로 범위 한정 (사전 없음)

# ─────────────────────────────────────────── 🤖 LLM (답변 생성) — 토큰 절약 핵심 설정
#   LLM_MODEL       : "gpt-5-mini"  (추론 모델, 입력 $0.25 · 출력 $2.00 / 1M 토큰)
#                     "gpt-6-luna"  (가장 싸고 빠른 GPT-6, 입력 $0.10 · 출력 $0.50 / 1M — REASONING_EFFORT="none" 이면 추론 없이 답변)
#   REASONING_EFFORT: none | minimal | low | medium | high | xhigh | max
#                     낮을수록 '생각(추론) 토큰'이 줄어 싸고 빠름. none = 추론 토큰 0 (gpt-6-luna 등 지원 모델만)
#                     모델이 지원하지 않는 값이면 가장 가까운 낮은 값으로 자동 조정 (예: gpt-5-mini 에 none → minimal, xhigh/max → high)
#   ※ 추론 토큰은 '출력 토큰'으로 과금됩니다. 추론 강도 none 일 때만 TEMPERATURE·TOP_P 가 적용됩니다.
LLM_MODEL = "gpt-5-mini"          # 또는 "gpt-6-luna"
REASONING_EFFORT = "low"  
TEMPERATURE      = 0.2
TOP_P            = 1.0
MAX_TOKENS       = 2048       # 답변 1개 출력 상한 (추론 토큰 포함) — 낮추면 비용 상한도 내려감
ALLOW_GENERAL    = True       # True: 문서에 없으면 '[문서 외 일반지식]' 표시 후 답변
HISTORY_TURNS    = 3          # 02 는 질문마다 새 대화라 영향 없음 (03 챗봇에서만 사용)

# ─────────────────────────────────────────── 🔎 ① 질문 해석 — GPT 가 질문을 '검색용 질문'으로 다시 씀 (오타·한영 자판·한자·영어·줄임말)
#   원래 질문 + 다시 쓴 질문 N개로 각각 검색 → RRF 로 합침
QUERY_REWRITE            = True
REWRITE_MODEL            = "gpt-5-mini"     # 예) "gpt-6-luna" (더 쌈 · REWRITE_REASONING_EFFORT="none" 가능)
REWRITE_REASONING_EFFORT = "low"            # ★ 질문을 해석할 때 얼마나 '생각'할지: none | minimal | low | medium | high | xhigh | max
                                            #   low = 빠르고 쌈(일반 오타·자판) · high = 꼬인 질문·후속 질문도 꼼꼼히 (추론 토큰↑ = 비용↑)
REWRITE_N                = 2                # 다시 쓴 검색 질문 개수
REWRITE_MAX_TOKENS       = 2048             # 해석 1회 출력 상한 (추론 토큰 포함 — high 면 넉넉히)
REWRITE_HISTORY_TURNS    = 2                # 후속 질문('그럼 그 사업은?') 해석에 참고할 직전 대화 턴 수 (03 챗봇)
ANSWER_WITH_REWRITE      = True             # ★ True: 답변 GPT 에도 '검색용 질문'(1순위)을 원래 질문과 함께 전달 → 오타·자판·줄임말·후속 질문을 바로잡은 뜻으로 답함
                                            #   False: 답변 GPT 에는 원래 질문만 (다시 쓴 질문은 검색에만)
# ─────────────────────────────────────────── 🎯 ③ 리랭커 — 후보를 질문과 1:1 로 다시 채점해 진짜 관련 있는 TOP_K 만 (내 GPU · API 비용 0)
RERANK            = True
RERANK_MODEL      = "BAAI/bge-reranker-v2-m3"   # 다국어(한국어 포함) CrossEncoder · 첫 실행 때 약 2.3GB 다운로드
RERANK_CANDIDATES = 30                          # 리랭커에 넣을 후보 수 (멀티 쿼리 RRF 상위 N개)

# ─────────────────────────────────────────── ⚖️ RAGAS 심판(Judge) — 평가 토큰의 대부분이 여기서 나감
JUDGE_MODEL            = "gpt-5-mini"     # 예) "gpt-6-luna" + JUDGE_REASONING_EFFORT="none" → 평가 비용 크게 절감
JUDGE_REASONING_EFFORT = "none"           # none | minimal | low | medium | high | xhigh | max (자동 조정 동일)
JUDGE_MAX_TOKENS       = 4096

# ─────────────────────────────────────────── 📊 평가 방식
#   EVAL_MODE = "free" : 토큰 0 — 골든 데이터셋(0-5)으로 로컬 계산 (Key Fact 답변/근거 · Semantic Sim)
#               "llm"  : RAGAS (심판 LLM 호출, 질문당 약 15~20회)
#               "both" : 둘 다
EVAL_MODE = "both"
RAGAS_CFG = dict(metrics=["faithfulness", "answer_relevancy", "context_precision", "answer_correctness"],
                 judge_model=JUDGE_MODEL, judge_reasoning_effort=JUDGE_REASONING_EFFORT, judge_max_tokens=JUDGE_MAX_TOKENS)
PRICES = {"gpt-5-mini": (0.25, 2.00), "gpt-5-nano": (0.05, 0.40), "gpt-6-luna": (0.10, 0.50), "gpt-5.6-luna": (0.20, 1.20), "gpt-4.1-mini": (0.40, 1.60), "gpt-4.1": (2.00, 8.00)}   # (입력, 출력) $/1M 토큰 — 비용 표시용, 가격이 바뀌면 수정
def usd(p_in, p_out, model):
    """토큰 → 달러 (추론 토큰은 출력 토큰에 포함되어 과금). 가격표에 없는 모델이면 None"""
    p = next((v for k, v in sorted(PRICES.items(), key=lambda kv: -len(kv[0])) if str(model).lower().startswith(k.lower())), None)
    return None if p is None else (p_in or 0) / 1e6 * p[0] + (p_out or 0) / 1e6 * p[1]
def fmt_usd(x):
    return "—" if x is None else (f"${x:,.4f}" if x >= 0.01 else f"${x:.6f}")

# ============ 적용: 01 의 build_config.json(파싱·청크·임베딩·FAISS 설정 + 인덱스 경로) + 위 값 ============
from rag.config import load_config, set_by_path
from rag.hw import profile_overrides, profile_name
from rag.pipeline import Pipeline
from rag import notebook_checkpoint as nb
pd.set_option("display.max_colwidth", 120)
CFG = load_config()
CFG["retrieval"].update(top_k=TOP_K, dense_weight=DENSE_WEIGHT, fusion=FUSION, candidate_k=CANDIDATE_K, rrf_k=RRF_K)
CFG["routing"].update(ROUTING)
CFG["llm"].update(model=LLM_MODEL, temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS,
                  reasoning_effort=REASONING_EFFORT, allow_general=ALLOW_GENERAL, history_turns=HISTORY_TURNS)
CFG["rules"]["enabled"] = False              # 규칙·사전 전처리 없음 — 질문 그대로
CFG["evaluation"].update(RAGAS_CFG)
CFG["retrieval"].update(query_rewrite=dict(enabled=QUERY_REWRITE, model=REWRITE_MODEL, effort=REWRITE_REASONING_EFFORT, n=REWRITE_N,
                                            history_turns=REWRITE_HISTORY_TURNS, answer_with_rewrite=ANSWER_WITH_REWRITE),
                        rerank=dict(enabled=RERANK, model=RERANK_MODEL, candidates=RERANK_CANDIDATES))   # 설정이 바뀌면(답변에 검색용 질문 반영 포함) 02 결과 폴더도 새로

# Windows(로컬)에서 만든 artifacts 를 GCP(리눅스)로 옮겨 온 경우: 저장된 경로의 '\' → '/' 로 바꿔야 파일을 찾음
def fix_windows_paths(work_dir):
    import json as _json
    if os.sep == "\\" or not Path(work_dir).exists():
        return 0
    n = 0
    def conv(x):
        nonlocal n
        if isinstance(x, str) and "\\" in x and ("artifacts" in x or x.startswith(str(work_dir))):
            n += 1
            return x.replace("\\", "/")
        if isinstance(x, dict):
            return {k: conv(v) for k, v in x.items()}
        if isinstance(x, list):
            return [conv(v) for v in x]
        return x
    for name in ("active.json", "build_config.json", "manifest.json"):
        p = Path(work_dir) / name
        if p.exists():
            d = _json.loads(p.read_text(encoding="utf-8"))
            d2 = conv(d)
            if d2 != d:
                p.write_text(_json.dumps(d2, ensure_ascii=False, indent=2), encoding="utf-8")
    return n
_n = fix_windows_paths(WORK_DIR)
if _n:
    print(f"ℹ️ Windows 에서 만든 artifacts 감지 → 경로 {_n}개를 리눅스 형식으로 변환했습니다")
pipe = Pipeline.from_build(CFG, WORK_DIR)    # ← 학습 결과 불러오기 (없으면 01 을 실행하라는 오류)
CFG = pipe.cfg
po = profile_overrides(HW)                   # 질문 임베딩 장치만 현재 PC 에 맞춤
CFG["embedding"].update(device=po["embedding.device"], fp16=po["embedding.fp16"])
CFG["faiss"]["hnsw_ef_search"] = HNSW_EF_SEARCH
bi = pipe.build_info
display(pd.DataFrame({"학습 저장 시각": bi["saved_at"], "임베딩 모델(인덱스)": bi["active"].get("embed_model"),
                      "청크 수": (bi["stats"].get("chunk") or {}).get("chunks"),
                      "chunk_size": bi["cfg"]["chunking"]["chunk_size"], "FAISS": bi["cfg"]["faiss"]["index_type"],
                      "HWP 로더": bi["cfg"]["parsing"].get("hwp_loader"), "generate_only": pipe.generate_only}, index=["01 학습 결과"]).T)
print(f"Dense {DENSE_WEIGHT:.0%} : BM25 {1 - DENSE_WEIGHT:.0%} · top-k {TOP_K} · 답변 {LLM_MODEL}(추론 {REASONING_EFFORT}) · max_tokens {MAX_TOKENS} · 심판 {JUDGE_MODEL}(추론 {JUDGE_REASONING_EFFORT}) · 평가 {EVAL_MODE}")
from rag.llm import effort_table
print(f"🔎 질문 해석 {'켜짐 · ' + REWRITE_MODEL + ' · 추론 ' + REWRITE_REASONING_EFFORT + f' · 검색 질문 {REWRITE_N}개' if QUERY_REWRITE else '꺼짐'}"
      f" · 🎯 리랭커 {RERANK_MODEL + f' (후보 {RERANK_CANDIDATES} → {TOP_K})' if RERANK else '꺼짐'}")
display(effort_table(([("검색 질문 해석", REWRITE_MODEL, REWRITE_REASONING_EFFORT)] if QUERY_REWRITE else [])
                     + [("답변 생성", LLM_MODEL, REASONING_EFFORT), ("RAGAS 심판", JUDGE_MODEL, JUDGE_REASONING_EFFORT)]))

,01 학습 결과
학습 저장 시각,2026-10-04 10:05:40
임베딩 모델(인덱스),nlpai-lab/KURE-v1
청크 수,18041
chunk_size,700
FAISS,flat
HWP 로더,HwpHwpxLoader
generate_only,True


Dense 80% : BM25 20% · top-k 5 · 답변 gpt-5-mini(추론 low) · max_tokens 2048 · 심판 gpt-5-mini(추론 none) · 평가 both
🔎 질문 해석 켜짐 · gpt-5-mini · 추론 low · 검색 질문 2개 · 🎯 리랭커 BAAI/bge-reranker-v2-m3 (후보 30 → 5)


,용도,모델,요청 추론 강도,실제 적용,추론 모델,지원 값,비고
0,검색 질문 해석,gpt-5-mini,low,low,예,"minimal, low, medium, high",
1,답변 생성,gpt-5-mini,low,low,예,"minimal, low, medium, high",
2,RAGAS 심판,gpt-5-mini,none,minimal,예,"minimal, low, medium, high","gpt-5-mini: 추론 강도 'none' 미지원 → 'minimal' 로 호출 (지원: minimal, low, medium, high)"


In [3]:
# 0-3) ★ 질문 — 사용자가 챗봇에 직접 입력한 문장 그대로 (전처리·가공·LLM 생성 없음). 줄을 추가·삭제·수정하세요.
QUESTIONS = [
    "월세액 세액공재 한도가 얼마에요?",
    "연말졍산은 언제 하나여?",
    "장기주택저당 차임금 이자 공제한도",
    "기부금이월공제기간몇년",
    "중퇴자 연정 언제함?",
    "월세공제 몇 퍼임?",
    "dnjftp rhdwp 한도 알려줘",
    "dusakfwjdtks 언제 해요?",
    "Monthly rent 세액공제 rate가 몇 %?",
    "외국인 flat tax 19% 고르면 deduction 못 받아요?",
    "When is the deadline for the earned income payment statement?",
    "月貰 稅額控除 限度는?",
    "寄附金 移越控除 期間은 몇 年?",
    "월세 세액공제율이 30%로 올랐다면서요?",
    "지급명세서는 5월 31일까지 내면 되죠?",
]

# 질문마다 독립 실행 — 앞 질문을 기억하지 않음(후속 질문·대화 이어가기 없음)
from rag.evaluate import load_eval_set
EVAL_ITEMS = load_eval_set([dict(id=f"Q{n:02d}", conv=f"chat{n}", set="질문", question=q)   # conv 를 질문마다 다르게 = 매번 새 대화
                            for n, q in enumerate(QUESTIONS, 1)])
print(f"질문 {len(EVAL_ITEMS)}개 (각각 독립 실행)")

질문 15개 (각각 독립 실행)


In [4]:
# # 0-4) ★ 정답(reference) — Answer Correctness 계산용. 질문 문장(0-3)은 건드리지 않고 '질문 → 정답'을 따로 적습니다.
# #   키는 0-3 의 질문 문장과 똑같아야 합니다. 정답이 없는 질문은 빼거나 "" 로 두면 Answer Correctness 가 '—'(계산 안 함).
# #   ※ 2025년 귀속(2026년 초 연말정산) 기준 예시입니다. 평가 전에 최신 국세청 안내로 확인·수정하세요.
# REFERENCES = {
#     '월세액 세액공재 한도가 얼마에요?':
#         '월세액 세액공제 대상 월세액 한도는 연 1,000만원입니다(2024년 귀속부터 750만원에서 1,000만원으로 상향).',
#     '연말졍산은 언제 하나여?':
#         '근로소득 연말정산은 다음 해 2월분 급여를 지급할 때(통상 2월 말까지) 회사(원천징수의무자)가 실시합니다. 간소화 자료는 1월 15일경부터 홈택스에서 조회할 수 있습니다.',
#     '장기주택저당 차임금 이자 공제한도':
#         '장기주택저당차입금 이자상환액 소득공제 한도는 상환기간 15년 이상이면서 고정금리·비거치식 분할상환이면 2,000만원, 고정금리 또는 비거치식이면 1,800만원, 그 밖의 대출은 800만원, 상환기간 10년 이상 고정금리 또는 비거치식이면 600만원입니다.',
#     '기부금이월공제기간몇년':
#         '한도 초과로 공제받지 못한 기부금은 다음 과세기간부터 10년 이내에 이월하여 공제받을 수 있습니다.',
#     '중퇴자 연정 언제함?':
#         '중도퇴직자는 퇴직하는 달의 급여를 지급할 때 회사가 연말정산을 합니다. 퇴직 정산 때 반영하지 못한 공제는 다음 해 5월 종합소득세 확정신고로 추가 공제받을 수 있습니다.',
#     '월세공제 몇 퍼임?':
#         '월세액 세액공제율은 총급여 5,500만원 이하(종합소득금액 4,500만원 이하)는 17%, 총급여 5,500만원 초과 8,000만원 이하는 15%입니다.',
#     'dnjftp rhdwp 한도 알려줘':
#         '월세액 세액공제 대상 월세액 한도는 연 1,000만원입니다.',
#     'dusakfwjdtks 언제 해요?':
#         '근로소득 연말정산은 다음 해 2월분 급여를 지급할 때(통상 2월 말까지) 회사가 실시합니다.',
#     'Monthly rent 세액공제 rate가 몇 %?':
#         '월세액 세액공제율은 총급여 5,500만원 이하 17%, 5,500만원 초과 8,000만원 이하 15%입니다.',
#     '외국인 flat tax 19% 고르면 deduction 못 받아요?':
#         '외국인근로자가 19% 단일세율을 선택하면 비과세·소득공제·세액공제 및 감면을 적용받을 수 없습니다. 단일세율은 국내에서 최초로 근로를 제공한 날부터 20년 이내에 받는 근로소득에 적용됩니다.',
#     'When is the deadline for the earned income payment statement?':
#         'The payment statement for earned income must be submitted by March 10 of the following year.',
#     '月貰 稅額控除 限度는?':
#         '월세액 세액공제 대상 월세액 한도는 연 1,000만원입니다.',
#     '寄附金 移越控除 期間은 몇 年?':
#         '기부금 이월공제 기간은 10년입니다.',
#     '월세 세액공제율이 30%로 올랐다면서요?':
#         '30%로 오른 사실은 없습니다. 월세액 세액공제율은 총급여 5,500만원 이하 17%, 5,500만원 초과 8,000만원 이하 15%입니다.',
#     '지급명세서는 5월 31일까지 내면 되죠?':
#         '아닙니다. 근로소득 지급명세서 제출기한은 다음 해 3월 10일입니다. 5월 31일은 종합소득세 확정신고 기한입니다.',
# }
# _noref = [it["question"] for it in EVAL_ITEMS if not REFERENCES.get(it["question"])]
# print(f"정답 있음 {len(EVAL_ITEMS) - len(_noref)}/{len(EVAL_ITEMS)}개" + (f" · 정답 없음(Answer Correctness 제외): {_noref}" if _noref else ""))

In [5]:
# 0-5) ⭐ 골든 데이터셋 — 질문(0-3) + 정답(0-4) + 핵심 사실 → eval/golden_set.jsonl  (LLM 호출 없음)
#   한 번 만들어 두면: EVAL_MODE="free" 로 토큰 0 평가 · 03 챗봇이 같은 질문에 정답을 자동으로 붙여 평가
#   KEY_FACTS: 답변에 꼭 들어가야 할 핵심 사실 ('A|B' = 둘 중 하나면 정답). 비워 두면 정답 문장에서 숫자·%·날짜를 자동 추출
# GOLDEN_PATH = "eval/golden_set.jsonl"
# USE_SAVED_GOLDEN = False     # True: 아래 대신 저장된 파일을 그대로 사용 (파일을 직접 고쳤을 때)
# KEY_FACTS = {
#     "월세액 세액공재 한도가 얼마에요?": ["1,000만"],
#     "연말졍산은 언제 하나여?": ["2월"],
#     "장기주택저당 차임금 이자 공제한도": ["2,000만", "1,800만", "800만", "600만"],
#     "기부금이월공제기간몇년": ["10년"],
#     "중퇴자 연정 언제함?": ["퇴직", "5월"],
#     "월세공제 몇 퍼임?": ["17%", "15%"],
#     "dnjftp rhdwp 한도 알려줘": ["1,000만"],
#     "dusakfwjdtks 언제 해요?": ["2월"],
#     "Monthly rent 세액공제 rate가 몇 %?": ["17%", "15%"],
#     "외국인 flat tax 19% 고르면 deduction 못 받아요?": ["19%", "비과세|공제"],
#     "When is the deadline for the earned income payment statement?": ["March 10|3월 10일"],
#     "月貰 稅額控除 限度는?": ["1,000만"],
#     "寄附金 移越控除 期間은 몇 年?": ["10년"],
#     "월세 세액공제율이 30%로 올랐다면서요?": ["15%", "17%"],
#     "지급명세서는 5월 31일까지 내면 되죠?": ["3월 10일"],
# }
# from rag.evaluate import build_golden, save_golden, load_golden
# GOLDEN = load_golden(GOLDEN_PATH) if USE_SAVED_GOLDEN else build_golden(QUESTIONS, REFERENCES, KEY_FACTS)
# if not USE_SAVED_GOLDEN:
#     save_golden(GOLDEN, GOLDEN_PATH)
# GOLD = {g["question"]: g for g in GOLDEN}
# display(pd.DataFrame([{"id": g["id"], "질문": g["question"], "정답": "✔" if g["reference"] else "—",
#                        "핵심 사실": " · ".join(g["key_facts"]), "출처": g["key_facts_source"]} for g in GOLDEN]))
# print(f"골든 데이터셋 {len(GOLDEN)}개 → {GOLDEN_PATH}")

## 1. Generate — 저장된 인덱스로 질문별 답변 (학습·임베딩 없음)
> 👥 **담당:** 1-1b ①④ 질문 해석 · 답변 (`rag/rewrite.py`) · ②③ 검색 · 리랭킹 (`rag/retrieval.py`) · 1-2 · 1-3 평가 · 데이터셋

In [6]:
# 1-1) 인덱스 로드 (FAISS + BM25 + 임베딩 모델 — 질문 임베딩용 추론만) + 생성기
from rag.llm import Generator
retriever = pipe.load_retriever()
generator = Generator(CFG["llm"]["model"], CFG["llm"].get("timeout", 120))
chat = pipe.make_chat(retriever, generator)
print("FAISS 벡터", retriever.store.index.ntotal, "| BM25 청크", len(retriever.by_id), "| 임베딩", pipe.active.get("embed_model"),
      "| 전처리:", "없음(질문 그대로)" if chat.pre is None else "켜짐", "|", gpu_mem())

FAISS 벡터 18041 | BM25 청크 18041 | 임베딩 nlpai-lab/KURE-v1 | 전처리: 없음(질문 그대로) | GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 11.1 / 22.0GB


In [7]:
# 1-1b) 🔎 검색 파이프라인 — ① GPT 질문 해석 → ② (원래 질문 + 다시 쓴 질문) 각각 BM25 + Dense → RRF 병합 → ③ 리랭커
#      → ④ 답변 GPT 에는 [원 질문] + [검색용 질문(GPT 해석 1순위)] 을 함께 보냅니다 (ANSWER_WITH_REWRITE=True)
#      코드는 rag/ 에 있음 — 이 셀은 0-2 설정값으로 조립만 합니다 (02·03 이 같은 코드를 씀)
#        rag/rewrite.py   ① 질문 해석 · ④ 답변에 검색용 질문 전달   (담당: 질문 해석 · 답변 · 챗봇)
#        rag/retrieval.py ② 멀티 쿼리 하이브리드 · ③ 리랭커            (담당: 검색 · 리랭킹)
import importlib
import rag.rewrite, rag.retrieval
importlib.reload(rag.rewrite); importlib.reload(rag.retrieval)     # .py 를 고친 뒤 이 셀만 다시 실행하면 바로 반영
from rag.rewrite import QueryRewriter, install_answer_with_rewrite
from rag.retrieval import SmartRetriever

RW = (QueryRewriter(REWRITE_MODEL, effort=REWRITE_REASONING_EFFORT, n=REWRITE_N, max_tokens=REWRITE_MAX_TOKENS,
                    history_turns=REWRITE_HISTORY_TURNS, price=usd) if QUERY_REWRITE else None)
SR = SmartRetriever(retriever, chat, rewriter=RW, rerank=RERANK, rerank_model=RERANK_MODEL, candidates=RERANK_CANDIDATES)
chat.ret = SR                                   # 이제 chat.ask() 의 검색 = ① → ② → ③
install_answer_with_rewrite(chat, SR, enabled=ANSWER_WITH_REWRITE)   # ④ 답변 GPT 에 검색용 질문 함께 전달

if RERANK:                                      # 디스크·RAM 여유를 확인하고 리랭커를 미리 올림 — 안 되면 리랭커 없이 계속
    _ok, _why = SR.prepare_reranker()
    if _ok:
        print(f"🎯 리랭커 준비: {RERANK_MODEL} ({_why}) · {gpu_mem()}")
    else:
        RERANK = False
        CFG["retrieval"]["rerank"]["enabled"] = False      # 결과 폴더 이름에도 '리랭커 끔'으로 반영
        print(f"⚠️ 리랭커 건너뜀 — {_why} → 리랭커 없이(질문 해석 + 멀티 쿼리 하이브리드) 계속합니다")
print(f"🔎 질문 해석: {'켜짐 · ' + REWRITE_MODEL + ' · 추론 ' + REWRITE_REASONING_EFFORT if QUERY_REWRITE else '꺼짐 (원래 질문만 검색)'}"
      f" · 검색 = 원래 질문 + 다시 쓴 질문 {REWRITE_N if QUERY_REWRITE else 0}개 → RRF → {'리랭커 → ' if RERANK else ''}TOP {TOP_K}")
print(f"🧠 답변 GPT 에 보내는 질문: {'원래 질문 + 검색용 질문(1순위)' if (QUERY_REWRITE and ANSWER_WITH_REWRITE) else '원래 질문만'}"
      + (' (질문 해석이 꺼져 있어 원래 질문만)' if (ANSWER_WITH_REWRITE and not QUERY_REWRITE) else ''))

# ── 15개 질문을 GPT 가 어떻게 해석하는지 먼저 확인 (결과는 저장 → 1-2 에서 같은 질문은 다시 호출하지 않음)
PREVIEW_REWRITE = True
RUN_DIR = pipe.eval_set_dir(EVAL_ITEMS)
os.makedirs(RUN_DIR, exist_ok=True)
SR.attach(os.path.join(RUN_DIR, "rewrites.jsonl"))
if QUERY_REWRITE and PREVIEW_REWRITE:
    _u0 = SR.usage()
    _prev = []
    for it in EVAL_ITEMS:
        rw = SR.rewrite(it["question"])
        _prev.append({"id": it["id"], "질문(입력 그대로)": it["question"],
                      "검색용 질문 (GPT 해석)": " · ".join(rw["queries"]) or f"⚠️ {rw.get('error') or '해석 결과 없음'} → 원래 질문만 검색",
                      "추론 강도": rw.get("effort") or "-", "입력/출력 토큰": f"{rw['prompt_tokens']:,} / {rw['completion_tokens']:,}",
                      "비용": fmt_usd(rw.get("usd")), "": "새로 호출" if rw["new"] else "저장본"})
    display(pd.DataFrame(_prev).set_index("id").style.set_properties(**{"text-align": "left"}))
    _u = SR.usage_since(_u0)
    print(f"💵 이 셀의 GPT 사용 — 질문 해석({REWRITE_MODEL}·{REWRITE_REASONING_EFFORT}) {_u['calls']}회 · 입력 {_u['prompt']:,} · "
          f"출력 {_u['completion']:,}(추론 {_u['reasoning']:,}) 토큰 → {fmt_usd(_u['usd'])}  (저장본 재사용분은 $0)")

⚠️ 리랭커 건너뜀 — RAM 여유 2.9GB — 리랭커를 올리려면 4GB 이상 필요 (다른 노트북 커널을 끄세요) → 리랭커 없이(질문 해석 + 멀티 쿼리 하이브리드) 계속합니다
🔎 질문 해석: 켜짐 · gpt-5-mini · 추론 low · 검색 = 원래 질문 + 다시 쓴 질문 2개 → RRF → TOP 5
🧠 답변 GPT 에 보내는 질문: 원래 질문 + 검색용 질문(1순위)


,질문(입력 그대로),검색용 질문 (GPT 해석),추론 강도,입력/출력 토큰,비용,
id,,,,,,
Q01,월세액 세액공재 한도가 얼마에요?,월세액 세액공제 한도가 얼마인가요? · 월세 세액공제 상한액은 얼마인가요?,low,296 / 235,$0.000544,저장본
Q02,연말졍산은 언제 하나여?,연말정산 일정(시행 시기) 언제인지 · 연말정산 접수 기간 및 제출 기한,low,293 / 169,$0.000411,저장본
Q03,장기주택저당 차임금 이자 공제한도,장기주택저당차입금 이자 공제한도 · 장기주택저당차입금 이자 공제 한도(소득세 관련),low,298 / 181,$0.000437,저장본
Q04,기부금이월공제기간몇년,기부금 이월공제 기간은 몇 년인가요? · 기부금 이월공제 기간이 몇 년인지 알려 주세요.,low,293 / 173,$0.000419,저장본
Q05,중퇴자 연정 언제함?,중퇴자 연말정산은 언제 하나? · 중퇴자 연말정산 신고 기간은 언제인가?,low,291 / 232,$0.000537,저장본
Q06,월세공제 몇 퍼임?,월세 소득공제 비율은 얼마인가요? · 월세 소득공제 적용 공제율은 얼마인가요?,low,291 / 172,$0.000417,새로 호출
Q07,dnjftp rhdwp 한도 알려줘,월세 공제 한도는 얼마인가? · 월세 소득공제 한도는 얼마인가?,low,293 / 870,$0.001813,새로 호출
Q08,dusakfwjdtks 언제 해요?,연말정산은 언제 시행하나요? · 연말정산 일정은 언제인가요?,low,293 / 804,$0.001681,새로 호출
Q09,Monthly rent 세액공제 rate가 몇 %?,월세 세액공제율이 몇 퍼센트인지 · 월세 세액공제 적용 비율이 얼마인지,low,294 / 170,$0.000414,새로 호출


💵 이 셀의 GPT 사용 — 질문 해석(gpt-5-mini·low) 10회 · 입력 2,956 · 출력 3,409(추론 2,880) 토큰 → $0.007557  (저장본 재사용분은 $0)


In [8]:
# 1-2) 질문별 답변 생성 (1건마다 저장 → ■ 후 재실행하면 이어서). 질문마다 독립 — 앞 질문을 기억하지 않음
#      검색 = 1-1b 파이프라인(질문 해석 → 멀티 쿼리 하이브리드 → 리랭커)
#      답변 = 답변 GPT 가 [원 질문] + [검색용 질문(GPT 해석 1순위)] 을 함께 받아 답함 (0-2 ANSWER_WITH_REWRITE)
SHOW_ANSWERS = True       # True: 답변 원문도 표시
import os
from IPython.display import Markdown
from rag.evaluate import run_eval_set
from rag.notebook_checkpoint import JsonlStore
RUN_DIR = pipe.eval_set_dir(EVAL_ITEMS)
answers, NEW_IDS = [], set()
_u0 = SR.usage()
h = display(Markdown("⏳ 답변 생성 시작…"), display_id=True)
try:
    with pipe.stop.install_signal_handlers():
        for row, new in run_eval_set(chat, EVAL_ITEMS, JsonlStore(os.path.join(RUN_DIR, "answers.jsonl"), key="id"), pipe.stop):
            answers.append(row)
            if new:
                NEW_IDS.add(row["id"])
            h.update(Markdown(f"답변 생성 **{len(answers)}/{len(EVAL_ITEMS)}** · 방금 {row['id']} ({'새로 생성' if new else '저장본'})"))
            if SHOW_ANSWERS:
                _rw = SR.rw_for(row["question"]) or {}
                _rq = " · ".join(_rw.get("queries") or []) or "(해석 없음 — 원래 질문만 검색)"
                _aq = (row.get("augmentation") or {}).get("답변에 쓴 질문")
                _al = (f"🧠 답변 GPT 가 받은 질문: [원 질문] {row['question']} ➕ [검색용 질문] **{_aq}**" if _aq and _aq != row["question"]
                       else "🧠 답변 GPT 가 받은 질문: 원래 질문만" + ("" if _aq else " (이전 방식 저장본)"))
                display(Markdown(f"**{row['id']}. {row['question']}**  ⏱ {row.get('total_s')}s (검색 {row.get('retrieval_s')}s · 생성 {row.get('llm_s')}s)"
                                 f"\n\n🔎 검색용 질문: {_rq}\n\n{_al}\n\n{row['answer']}\n\n---"))
except KeyboardInterrupt:
    pass
if len(answers) < len(EVAL_ITEMS):
    h.update(Markdown(f"⏸ **중지됨** {len(answers)}/{len(EVAL_ITEMS)} — 셀을 다시 실행하면 이어서 진행합니다."))
else:
    h.update(Markdown(f"✅ 답변 {len(answers)}개 완료 → 아래 2번에서 질문별 RAGAS"))
_nq = sum(1 for r in answers if (r.get("augmentation") or {}).get("검색용 질문 반영"))
print(f"🧠 검색용 질문을 답변에 반영: {_nq}/{len(answers)}건" + ("" if ANSWER_WITH_REWRITE else " (ANSWER_WITH_REWRITE=False — 원래 질문만)"))

# 💵 이 셀의 GPT 사용 (이번 실행에서 새로 호출한 것만 — 저장본 재사용분은 $0)
def _aug(r, k):
    return (r.get("augmentation") or {}).get(k) or 0
_u = SR.usage_since(_u0)
_new = [r for r in answers if r["id"] in NEW_IDS]
ANS_NEW = dict(calls=len(_new), prompt=sum(_aug(r, "prompt_tokens") for r in _new), completion=sum(_aug(r, "completion_tokens") for r in _new),
               reasoning=sum(_aug(r, "reasoning_tokens") for r in _new))
ANS_NEW["usd"] = usd(ANS_NEW["prompt"], ANS_NEW["completion"], LLM_MODEL) or 0
print(f"💵 이 셀의 GPT 사용\n"
      f"   ① 질문 해석({REWRITE_MODEL}·{REWRITE_REASONING_EFFORT}) {_u['calls']}회 · 입력 {_u['prompt']:,} · 출력 {_u['completion']:,}(추론 {_u['reasoning']:,}) → {fmt_usd(_u['usd'])}"
      + ("  (1-1b 에서 이미 해석한 질문은 $0)" if QUERY_REWRITE else "") + "\n"
      f"   ④ 답변 생성({LLM_MODEL}·{REASONING_EFFORT}) {ANS_NEW['calls']}회 · 입력 {ANS_NEW['prompt']:,} · 출력 {ANS_NEW['completion']:,}(추론 {ANS_NEW['reasoning']:,}) → {fmt_usd(ANS_NEW['usd'])}\n"
      f"   합계 {fmt_usd(_u['usd'] + ANS_NEW['usd'])}  (저장본 재사용 {len(answers) - len(_new)}건은 $0)")

✅ 답변 15개 완료 → 아래 2번에서 질문별 RAGAS

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

**Q01. 월세액 세액공재 한도가 얼마에요?**  ⏱ 27.636s (검색 23.104s · 생성 4.53s)

🔎 검색용 질문: 월세액 세액공제 한도가 얼마인가요? · 월세 세액공제 상한액은 얼마인가요?

🧠 답변 GPT 가 받은 질문: [원 질문] 월세액 세액공재 한도가 얼마에요? ➕ [검색용 질문] **월세액 세액공제 한도가 얼마인가요?**

문서(제공된 컨텍스트)에는 '월세액 세액공제 한도'에 관한 내용이 없습니다.

일반 지식으로 안내받기를 원하시면 말씀해 주세요. 그 경우 답변 문장 앞에 '[문서 외 일반지식]'을 붙여 제공하겠습니다.

---

**Q02. 연말졍산은 언제 하나여?**  ⏱ 4.535s (검색 0.274s · 생성 4.26s)

🔎 검색용 질문: 연말정산 일정(시행 시기) 언제인지 · 연말정산 접수 기간 및 제출 기한

🧠 답변 GPT 가 받은 질문: [원 질문] 연말졍산은 언제 하나여? ➕ [검색용 질문] **연말정산 일정(시행 시기) 언제인지**

문서 내에 연말정산의 시행 시기(일정)는 명시되어 있지 않습니다.

[문서 외 일반지식] 한국의 통상적인 연말정산 일정은 다음과 같습니다.
- 근로자들이 소득·세액공제 증빙자료를 회사에 제출하는 기간: 보통 매년 1월 중순부터 2월 중순까지.  
- 사업주(회사)가 연말정산을 실시하고 결과를 반영하는 시기: 통상 1월 말~2월 중순에 연말정산을 완료하여 2월 급여에 반영하거나 3월 급여에서 정산하는 경우도 있음.  
- 국세청 신고·제출(원천징수 및 전산매체 등)은 연말정산 완료 직후(주로 2월 말까지) 이루어지는 경우가 많음.

상기 일정은 기관·기업별로 차이가 있을 수 있으니, 정확한 시행일정은 소속 기관의 공지나 인사·세무 담당부서 안내를 확인하시기 바랍니다.

---

**Q03. 장기주택저당 차임금 이자 공제한도**  ⏱ 4.466s (검색 0.304s · 생성 4.16s)

🔎 검색용 질문: 장기주택저당차입금 이자 공제한도 · 장기주택저당차입금 이자 공제 한도(소득세 관련)

🧠 답변 GPT 가 받은 질문: [원 질문] 장기주택저당 차임금 이자 공제한도 ➕ [검색용 질문] **장기주택저당차입금 이자 공제한도**

문서(컨텍스트)에는 "장기주택저당차입금 이자 공제한도"에 관한 내용이 없습니다. 먼저 해당 문서에서 관련 정보를 찾을 수 없음을 알려드립니다. 

원하시면 다음 중 하나를 선택해 주세요.
1) 어느 나라(예: 대한민국)·어느 과세연도 기준인지 알려주시면 그 기준에 따라 최신 제도를 찾아 정리해 드리겠습니다.  
2) 일반적(문서 외 일반지식)으로 장기주택저당차입금 이자공제의 개념·대상·일반적 적용방식(수치 제외)을 설명해 드리겠습니다.  

어떤 방식으로 답변해 드릴까요?

---

...

**Q12. 月貰 稅額控除 限度는?**  ⏱ 4.651s (검색 0.452s · 생성 4.2s)

🔎 검색용 질문: 월세 세액공제 한도는? · 월세에 대한 세액공제 한도 금액은 얼마인가요?

🧠 답변 GPT 가 받은 질문: [원 질문] 月貰 稅額控除 限度는? ➕ [검색용 질문] **월세 세액공제 한도는?**

컨텍스트 문서들에 '월세 세액공제 한도'에 관한 내용이 없습니다. 먼저 그 사실을 알려드립니다.

[문서 외 일반지식] 월세 세액공제(월세액에 대한 소득세액공제)에 대한 최신 정보 제공을 원하시면 알려주세요. 제공 시 연간 한도·공제율·적용대상(소득요건, 세대주 여부 등)을 포함해 설명드리겠습니다. 어떤 범위(예: 근로소득자 기준만, 또는 사업소득자 포함 등)를 원하시는지도 알려주시면 맞춰서 답변드리겠습니다.

---

**Q13. 寄附金 移越控除 期間은 몇 年?**  ⏱ 3.119s (검색 0.24s · 생성 2.88s)

🔎 검색용 질문: 기부금의 이월공제 기간은 몇 년인가? · 기부금 이월공제의 적용 기간은 몇 년인지

🧠 답변 GPT 가 받은 질문: [원 질문] 寄附金 移越控除 期間은 몇 年? ➕ [검색용 질문] **기부금의 이월공제 기간은 몇 년인가?**

문서들에 기부금의 이월공제(이월공제 기간)에 관한 내용은 포함되어 있지 않습니다. 먼저 이 점을 알려드립니다.

원하시면 일반 세법상 규정(문서 외 일반지식)을 찾아 설명해 드리겠습니다. 계속 진행할까요?

---

**Q14. 월세 세액공제율이 30%로 올랐다면서요?**  ⏱ 7.486s (검색 0.389s · 생성 7.1s)

🔎 검색용 질문: 월세 세액공제율이 30%로 인상되었는지 여부와 인상 관련 근거(법령·시행령·시행규칙·행정해석), 시행일 및 적용 대상과 요건 · 월세 세액공제율 30% 관련 공고·보도자료·질의회신 확인: 시행 시점, 적용 대상(소득·세액 기준 등), 공제 한도 및 적용 절차

🧠 답변 GPT 가 받은 질문: [원 질문] 월세 세액공제율이 30%로 올랐다면서요? ➕ [검색용 질문] **월세 세액공제율이 30%로 인상되었는지 여부와 인상 관련 근거(법령·시행령·시행규칙·행정해석), 시행일 및 적용 대상과 요건**

컨텍스트 문서들에는 월세 세액공제율 관련 내용이 전혀 포함되어 있지 않습니다. 따라서 제공하신 문서들만으로는 “월세 세액공제율이 30%로 인상되었는지” 여부와 관련 근거·시행일·적용대상·요건을 확인할 수 없습니다.

[문서 외 일반지식] 정확한 최신 내용(세율·시행일·적용대상·요건 등)은 관련 법령 및 행정해석을 직접 확인해야 합니다. 확인해야 할 법령·자료와 확인 방법은 다음과 같습니다.
- 확인할 법령·자료
  - 소득세법(임대인·임차인 관련 조문 포함), 소득세법 시행령, 소득세법 시행규칙
  - 기획재정부·국세청 고시·예규·행정해석(국세청 홈택스·국세청 누리집의 고시·훈령·해석 사례)
  - 국무회의 의결문·공포령·관보(법령 공포일·시행일 확인용)
- 확인 방법(권장 순서)
  1. 법제처 국가법령정보센터에서 최신 「소득세법」·시행령·시행규칙 원문 검색 — 관련 조항(예: 주거 관련 세액공제 조항) 및 개정 이력·시행일 확인.
  2. 국세청(홈택스)·기획재정부 공지·해설 자료 확인 — 개정 내용의 적용대상·요건·증빙서류(임대차계약서, 계좌이체 영수증 등) 안내 확인.
  3. 관보·공포문 확인 — 법률·시행령의 공포일 및 시행일 명시.
  4. 필요시 지방국세청 또는 세무사에 문의하여 사례 적용 확인.
- 자주 확인해야 할 항목
  - 세액공제율(예: 30% 등) 및 적용기간(개정의 시행일)
  - 대상(무주택 세대주 여부, 근로소득자·연금소득자 등)
  - 소득·임대료 한도(공제한도 금액)
  - 증빙요건(계약서·계좌이체 등)
  - 소급적용 여부 또는 개정 이전 계약에 대한 적용

원하시면 제가 법령·국세청 공지 등을 실제로 찾아 최신 근거(법조문·시행령 조항·국세청 고시 등)를 검색해 드리거나, 사용자가 제공한 법령 전문을 근거로 문구·시행일·적용대상·요건을 정리해 드리겠습니다. 어느 방식으로 도와드릴까요?

---

**Q15. 지급명세서는 5월 31일까지 내면 되죠?**  ⏱ 3.482s (검색 0.264s · 생성 3.22s)

🔎 검색용 질문: 지급명세서 제출 기한이 5월 31일까지인지 확인해 주세요 · 지급명세서 제출 마감일이 5월 31일인지 여부

🧠 답변 GPT 가 받은 질문: [원 질문] 지급명세서는 5월 31일까지 내면 되죠? ➕ [검색용 질문] **지급명세서 제출 기한이 5월 31일까지인지 확인해 주세요**

문서에 지급명세서 제출기한(예: 5월 31일)에 관한 명시가 없습니다. 해당 제출기한은 확인할 수 없습니다. [1]

참고로 본 컨텍스트에서 제출기한 관련 기재는 다음과 같습니다.
- 입찰서 제출기한은 p.4의 「추진 일정」 및 입찰 공고문을 참조하라고 되어 있습니다. [1]
- 제안서 제출기한은 입찰서 제출기한과 동일하다고 명시되어 있습니다. [1]
- 일부 증빙서류는 “최근 3개월 이내” 발급분을 첨부하도록 요구하고 있습니다(사업자등록증 등). [5]

따라서 지급명세서의 제출기한이 5월 31일인지 확정하려면 해당 사업의 추진 일정(p.4) 또는 입찰 공고문을 직접 확인하시기 바랍니다. [1]

---

🧠 검색용 질문을 답변에 반영: 15/15건
💵 이 셀의 GPT 사용
   ① 질문 해석(gpt-5-mini·low) 0회 · 입력 0 · 출력 0(추론 0) → $0.000000  (1-1b 에서 이미 해석한 질문은 $0)
   ④ 답변 생성(gpt-5-mini·low) 15회 · 입력 26,970 · 출력 6,887(추론 4,096) → $0.0205
   합계 $0.0205  (저장본 재사용 0건은 $0)


In [9]:
# 1-3) ⏱ 질문별 응답 시간 + 🪙 토큰 + 💵 비용 — ① 질문 해석 · ④ 답변 생성 (저장본은 처음 생성했을 때 값)
TIME_COLS = ["검색(s)", "생성(s)", "전체 응답(s)"]
TOK_COLS = ["해석 토큰", "답변 입력", "답변 출력", "답변 추론"]
USD_COLS = ["해석 $", "답변 $", "합계 $"]
def _aug(r, k):
    return (r.get("augmentation") or {}).get(k)
_rows = []
for r in answers:
    rw = SR.rw_for(r["question"]) or {}
    a_usd = usd(_aug(r, "prompt_tokens") or 0, _aug(r, "completion_tokens") or 0, LLM_MODEL)
    _rows.append({"id": r["id"], "질문": r["question"], "검색용 질문": " · ".join(rw.get("queries") or []),
                  "답변 GPT 질문": (_aug(r, "답변에 쓴 질문") or r["question"]) if _aug(r, "검색용 질문 반영") else "원래 질문만",
                  "검색(s)": r.get("retrieval_s"), "생성(s)": r.get("llm_s"), "전체 응답(s)": r.get("total_s"),
                  "해석 토큰": (rw.get("prompt_tokens") or 0) + (rw.get("completion_tokens") or 0) if rw else None,
                  "답변 입력": _aug(r, "prompt_tokens"), "답변 출력": _aug(r, "completion_tokens"), "답변 추론": _aug(r, "reasoning_tokens"),
                  "해석 $": rw.get("usd") if rw else None, "답변 $": a_usd,
                  "합계 $": (rw.get("usd") or 0) + (a_usd or 0) if (rw or a_usd is not None) else None})
rt = pd.DataFrame(_rows)
_sty = (rt.style.format("{:.2f}", subset=TIME_COLS, na_rep="—").format("{:,.0f}", subset=TOK_COLS, na_rep="—")
        .format(lambda x: fmt_usd(x) if x == x and x is not None else "—", subset=USD_COLS)
        .set_properties(subset=["질문", "검색용 질문", "답변 GPT 질문"], **{"text-align": "left"}))
for _c, _col in (("전체 응답(s)", "#f4a582"), ("합계 $", "#92c5de")):
    if pd.to_numeric(rt[_c], errors="coerce").fillna(0).max() > 0:      # 전부 0 이면 막대 생략 (0 나누기 경고 방지)
        _sty = _sty.bar(subset=[_c], color=_col, vmin=0)
display(_sty)
_num = rt[TIME_COLS + TOK_COLS + USD_COLS].apply(pd.to_numeric, errors="coerce")
rt_stat = _num.agg(["mean", "median", "min", "max", "sum"]).rename(index={"mean": "평균", "median": "중앙값", "min": "최소", "max": "최대", "sum": "합계"})
display(rt_stat.style.format("{:.2f}", subset=TIME_COLS).format("{:,.0f}", subset=TOK_COLS).format(fmt_usd, subset=USD_COLS))
if len(rt):
    slow = rt.loc[pd.to_numeric(rt["전체 응답(s)"], errors="coerce").idxmax()]
    print(f"가장 느린 질문: {slow['id']} {slow['질문']} — {slow['전체 응답(s)']:.2f}s  ·  CPU {HW.get('cpu_model')} · GPU {HW.get('gpu') or '없음'}")
    print(f"💵 질문 {len(rt)}개 전체(저장본 포함): 질문 해석 {fmt_usd(_num['해석 $'].sum())} + 답변 생성 {fmt_usd(_num['답변 $'].sum())} = {fmt_usd(_num['합계 $'].sum())}"
          f"  · 질문당 평균 {fmt_usd(_num['합계 $'].mean())}   (이 셀 자체는 GPT 를 호출하지 않음 · $0)")

,id,질문,검색용 질문,답변 GPT 질문,검색(s),생성(s),전체 응답(s),해석 토큰,답변 입력,답변 출력,답변 추론,해석 $,답변 $,합계 $
0,Q01,월세액 세액공재 한도가 얼마에요?,월세액 세액공제 한도가 얼마인가요? · 월세 세액공제 상한액은 얼마인가요?,월세액 세액공제 한도가 얼마인가요?,23.10,4.53,27.64,531,"1,826",395,320,$0.000544,$0.001246,$0.001790
1,Q02,연말졍산은 언제 하나여?,연말정산 일정(시행 시기) 언제인지 · 연말정산 접수 기간 및 제출 기한,연말정산 일정(시행 시기) 언제인지,0.27,4.26,4.54,462,"2,370",438,192,$0.000411,$0.001469,$0.001880
2,Q03,장기주택저당 차임금 이자 공제한도,장기주택저당차입금 이자 공제한도 · 장기주택저당차입금 이자 공제 한도(소득세 관련),장기주택저당차입금 이자 공제한도,0.30,4.16,4.47,479,"1,721",422,256,$0.000437,$0.001274,$0.001711
3,Q04,기부금이월공제기간몇년,기부금 이월공제 기간은 몇 년인가요? · 기부금 이월공제 기간이 몇 년인지 알려 주세요.,기부금 이월공제 기간은 몇 년인가요?,0.24,2.45,2.68,466,"1,554",259,192,$0.000419,$0.000906,$0.001326
4,Q05,중퇴자 연정 언제함?,중퇴자 연말정산은 언제 하나? · 중퇴자 연말정산 신고 기간은 언제인가?,중퇴자 연말정산은 언제 하나?,0.21,5.13,5.34,523,"1,831",633,448,$0.000537,$0.001724,$0.002260
5,Q06,월세공제 몇 퍼임?,월세 소득공제 비율은 얼마인가요? · 월세 소득공제 적용 공제율은 얼마인가요?,월세 소득공제 비율은 얼마인가요?,0.20,3.55,3.75,463,"1,926",411,256,$0.000417,$0.001303,$0.001720
6,Q07,dnjftp rhdwp 한도 알려줘,월세 공제 한도는 얼마인가? · 월세 소득공제 한도는 얼마인가?,월세 공제 한도는 얼마인가?,0.18,3.62,3.81,"1,163","1,853",469,320,$0.001813,$0.001401,$0.003215
7,Q08,dusakfwjdtks 언제 해요?,연말정산은 언제 시행하나요? · 연말정산 일정은 언제인가요?,연말정산은 언제 시행하나요?,0.18,4.92,5.10,"1,097","2,090",522,320,$0.001681,$0.001566,$0.003248
8,Q09,Monthly rent 세액공제 rate가 몇 %?,월세 세액공제율이 몇 퍼센트인지 · 월세 세액공제 적용 비율이 얼마인지,월세 세액공제율이 몇 퍼센트인지,0.21,2.52,2.73,464,"1,667",222,128,$0.000414,$0.000861,$0.001274
9,Q10,외국인 flat tax 19% 고르면 deduction 못 받아요?,외국인이 단일세율 19%를 선택하면 공제를 받을 수 없나요? · 외국인이 단일세율 19%를 선택할 경우 세액공제 또는 소득공제를 적용받을 수 있는지?,외국인이 단일세율 19%를 선택하면 공제를 받을 수 없나요?,0.31,4.72,5.03,555,"1,726",470,128,$0.000589,$0.001371,$0.001960


,검색(s),생성(s),전체 응답(s),해석 토큰,답변 입력,답변 출력,답변 추론,해석 $,답변 $,합계 $
평균,1.79,4.06,5.84,588,"1,798",459,273,$0.000660,$0.001368,$0.002028
중앙값,0.25,4.16,4.47,523,"1,760",428,256,$0.000537,$0.001371,$0.001790
최소,0.18,2.45,2.68,406,"1,416",222,128,$0.000298,$0.000861,$0.001274
최대,23.10,7.10,27.64,"1,163","2,370",953,448,$0.001813,$0.002260,$0.003248
합계,26.81,60.86,87.67,"8,826","26,970","6,887","4,096",$0.009905,$0.0205,$0.0304


가장 느린 질문: Q01 월세액 세액공재 한도가 얼마에요? — 27.64s  ·  CPU Intel(R) Xeon(R) CPU @ 2.20GHz · GPU NVIDIA L4
💵 질문 15개 전체(저장본 포함): 질문 해석 $0.009905 + 답변 생성 $0.0205 = $0.0304  · 질문당 평균 $0.002028   (이 셀 자체는 GPT 를 호출하지 않음 · $0)


## 2. 평가 — ① 전체 질문 종합 → ② 질문별 (실시간 표)
> 👥 **담당:** 평가 · 데이터셋 (02 노트북 책임자) — `rag/evaluate.py` `qa_gen.py` `async_utils.py`
- `EVAL_MODE="free"`: **토큰 0** — 골든 데이터셋(0-5)으로 로컬 계산
  - **Key Fact (답변)**: 핵심 사실이 답변에 들어 있는 비율 (≈ 정답 여부)
  - **Key Fact (근거)**: 핵심 사실이 검색된 문단에 들어 있는 비율 (≈ 검색 품질)
  - **Semantic Sim**: 답변과 정답의 의미 유사도 (임베딩 코사인, 내 GPU/CPU)
- `EVAL_MODE="llm"`: RAGAS — Faithfulness · Answer Relevancy · Context Precision · Answer Correctness (심판 LLM 호출)
- `EVAL_MODE="both"`: 둘 다. 표 아래에 **RAGAS 심판이 쓴 토큰과 예상 비용**이 표시됩니다.
- 질문 1건·지표 1개가 끝날 때마다 표가 갱신되고 저장됩니다. ■ 후 다시 실행하면 남은 것만 계산합니다. 같은 설정이면 저장된 점수를 재사용(토큰 0)합니다.
- ② 질문별 표와 `ragas_per_question.csv` 에 응답 시간·토큰이 함께 들어갑니다.

In [10]:
# 2) 평가 — ① 전체 질문 종합 → ② 질문별.   EVAL_MODE: "free"(토큰 0) | "llm"(RAGAS) | "both"
import warnings
warnings.filterwarnings("ignore", message="Unexpected type for token usage")   # langchain-openai 경고(결과와 무관) 숨김
import rag.evaluate as EV
from rag.evaluate import score_rows_iter, free_metrics, FREE_LABELS
from rag.notebook_checkpoint import fingerprint
from rag import llm as L

assert EVAL_MODE in ("free", "llm", "both"), "EVAL_MODE 는 free | llm | both"
# 0-4(정답)·0-5(골든) 셀을 주석 처리하거나 건너뛰어도 실행되도록: 없으면 빈 dict
REFERENCES = globals().get("REFERENCES") or {}
GOLD = globals().get("GOLD") or {}
if not GOLD and EVAL_MODE == "both":
    print("ℹ️ 골든 데이터셋 없음 → 토큰 0 지표(Key Fact·Semantic Sim) 생략 · RAGAS 만 계산")
    EVAL_MODE = "llm"
if not GOLD and EVAL_MODE == "free":
    print("ℹ️ 골든 데이터셋 없음 → 'free' 로 계산할 지표가 없어 평가를 건너뜁니다(토큰 0). RAGAS 를 보려면 0-2 에서 EVAL_MODE='llm'")
if not REFERENCES and not GOLD:
    print("ℹ️ 정답 없음 → Answer Correctness 는 '—'(계산 안 함), Context Precision 은 정답 없는 방식(no-ref)으로 계산")
names = RAGAS_CFG["metrics"] if EVAL_MODE in ("llm", "both") else []
cols_llm = [EV.METRIC_LABELS[m] for m in names]
cols_free = list(FREE_LABELS.values()) if (EVAL_MODE == "both" or (EVAL_MODE == "free" and GOLD)) else []
cols = cols_free + cols_llm
qtext = {r["id"]: r["question"] for r in answers}
rtime = {r["id"]: r.get("total_s") for r in answers}
ref_of = lambda q: (GOLD.get(q) or {}).get("reference") or REFERENCES.get(q) or None   # noqa: E731

# ── 토큰 0 지표 (골든 데이터셋 · 로컬 계산)
free_rows = {}
if cols_free:
    for r in answers:
        g = GOLD.get(r["question"], {})
        free_rows[r["id"]] = free_metrics(r["answer"], r.get("contexts") or [], g.get("reference"), g.get("key_facts"),
                                          retriever.embedder)

def build_df(llm_df=None):
    d = pd.DataFrame([{"id": r["id"], "질문": r["question"], **free_rows.get(r["id"], {})} for r in answers])
    for c_ in cols_free:
        d[c_] = d.get(c_)
    if cols_llm:
        if llm_df is not None and len(llm_df):
            d = d.merge(llm_df[["id"] + cols_llm], on="id", how="left")
        else:
            for c_ in cols_llm:
                d[c_] = None
    d["응답(s)"] = d["id"].map(rtime)
    return d[["id", "질문"] + cols + ["응답(s)"]]

def overall(df):
    recs = []
    for c_ in cols:
        v = pd.to_numeric(df[c_], errors="coerce").dropna()
        recs.append({"지표": c_, "평균": v.mean() if len(v) else None, "중앙값": v.median() if len(v) else None,
                     "최소": v.min() if len(v) else None, "최대": v.max() if len(v) else None,
                     "계산된 질문": f"{len(v)}/{len(answers)}", "종류": "토큰 0" if c_ in cols_free else "RAGAS(LLM)"})
    means = [r_["평균"] for r_ in recs if r_["평균"] is not None and r_["평균"] == r_["평균"]]
    recs.append({"지표": "★ 종합 (지표 평균)", "평균": sum(means) / len(means) if means else None, "중앙값": None,
                 "최소": None, "최대": None, "계산된 질문": f"{len(means)}/{len(cols)} 지표", "종류": ""})
    return pd.DataFrame(recs).astype({"평균": float, "중앙값": float, "최소": float, "최대": float})

num = ["평균", "중앙값", "최소", "최대"]
def show(df, summ):
    h_sum.update(summ.style.format("{:.3f}", subset=num, na_rep="").background_gradient(subset=["평균"], cmap="RdYlGn", vmin=0, vmax=1)
                 .set_properties(subset=["지표"], **{"text-align": "left"}))
    h_tab.update(df.style.format("{:.3f}", subset=cols, na_rep="—").format("{:.2f}", subset=["응답(s)"], na_rep="—")
                 .background_gradient(subset=cols, cmap="RdYlGn", vmin=0, vmax=1).set_properties(subset=["질문"], **{"text-align": "left"}))

display(Markdown(f"### ① 전체 질문 종합 ({EVAL_MODE})"))
h_sum = display(pd.DataFrame(), display_id=True)
display(Markdown("### ② 질문별"))
h_status = display(Markdown("⏳ 준비 중…"), display_id=True)
h_tab = display(pd.DataFrame(), display_id=True)
ragas_df = build_df()
ragas_summary = overall(ragas_df)
show(ragas_df, ragas_summary)
L.reset_judge_usage()
if names:
    rows_for_eval = [{**r, "reference": ref_of(r["question"])} for r in answers]   # 질문은 그대로, 정답만 붙여 평가
    store_r = JsonlStore(os.path.join(RUN_DIR, f"ragas_{fingerprint(names, CFG['evaluation'], {r['question']: ref_of(r['question']) or '' for r in answers})}.jsonl"), key="id")
    try:
        with pipe.stop.install_signal_handlers():
            for df, _, status in score_rows_iter(rows_for_eval, names, CFG["evaluation"], retriever.embedder, store_r, pipe.stop):
                h_status.update(Markdown(f"**{status}** · 심판 호출 {L.JUDGE_USAGE['calls']}회"))
                if len(df):
                    ragas_df = build_df(df)
                    ragas_summary = overall(ragas_df)
                    show(ragas_df, ragas_summary)
    except KeyboardInterrupt:
        h_status.update(Markdown("⏸ **중지됨** — 셀을 다시 실행하면 남은 질문만 평가합니다."))
else:
    h_status.update(Markdown("✅ 토큰 0 평가 완료 (RAGAS 심판 호출 없음)"))

ju = dict(L.JUDGE_USAGE)
JUDGE_NEW = dict(calls=ju["calls"], prompt=ju["prompt_tokens"], completion=ju["completion_tokens"], reasoning=ju["reasoning_tokens"],
                 usd=usd(ju["prompt_tokens"], ju["completion_tokens"], JUDGE_MODEL) or 0)
print(f"💵 이 셀의 GPT 사용 — RAGAS 심판({JUDGE_MODEL}·{JUDGE_REASONING_EFFORT}) {JUDGE_NEW['calls']}회 · 입력 {JUDGE_NEW['prompt']:,} · "
      f"출력 {JUDGE_NEW['completion']:,}(추론 {JUDGE_NEW['reasoning']:,}) 토큰 → {fmt_usd(JUDGE_NEW['usd'])}  (저장된 점수 재사용분은 $0)")

out_csv = Path(WORK_DIR) / "ragas_per_question.csv"
ragas_df.drop(columns=["응답(s)"]).merge(pd.DataFrame([{
    "id": r["id"], "검색(s)": r.get("retrieval_s"), "생성(s)": r.get("llm_s"), "전체 응답(s)": r.get("total_s"),
    "입력 토큰": (r.get("augmentation") or {}).get("prompt_tokens"), "출력 토큰": (r.get("augmentation") or {}).get("completion_tokens"),
    "추론 토큰": (r.get("augmentation") or {}).get("reasoning_tokens"), "답변": r["answer"], "정답": ref_of(r["question"]) or "",
    "핵심 사실": " · ".join((GOLD.get(r["question"]) or {}).get("key_facts") or []),
    "검색용 질문": " · ".join((SR.rw_for(r["question"]) or {}).get("queries") or []),
    "답변 GPT 질문": ((r.get("augmentation") or {}).get("답변에 쓴 질문") or "") if (r.get("augmentation") or {}).get("검색용 질문 반영") else "원래 질문만",
    "해석 $": (SR.rw_for(r["question"]) or {}).get("usd"),
    "답변 $": usd((r.get("augmentation") or {}).get("prompt_tokens") or 0, (r.get("augmentation") or {}).get("completion_tokens") or 0, LLM_MODEL)} for r in answers]),
    on="id", how="left").to_csv(out_csv, index=False, encoding="utf-8-sig")
ragas_summary.to_csv(Path(WORK_DIR) / "ragas_summary.csv", index=False, encoding="utf-8-sig")
print("저장:", out_csv, "·", Path(WORK_DIR) / "ragas_summary.csv")

ℹ️ 골든 데이터셋 없음 → 토큰 0 지표(Key Fact·Semantic Sim) 생략 · RAGAS 만 계산
ℹ️ 정답 없음 → Answer Correctness 는 '—'(계산 안 함), Context Precision 은 정답 없는 방식(no-ref)으로 계산


### ① 전체 질문 종합 (llm)

,지표,평균,중앙값,최소,최대,계산된 질문,종류
0,Faithfulness,0.419,0.400,0.111,1.000,15/15,RAGAS(LLM)
1,Answer Relevancy,0.203,0.000,0.000,0.796,15/15,RAGAS(LLM)
2,Context Precision,0.561,0.500,0.000,1.000,15/15,RAGAS(LLM)
3,Answer Correctness,,,,,0/15,RAGAS(LLM)
4,★ 종합 (지표 평균),0.394,,,,3/4 지표,


### ② 질문별

**✅ 완료 15개** · 심판 호출 150회

,id,질문,Faithfulness,Answer Relevancy,Context Precision,Answer Correctness,응답(s)
0,Q01,월세액 세액공재 한도가 얼마에요?,0.333,0.000,0.750,—,27.64
1,Q02,연말졍산은 언제 하나여?,0.400,0.672,0.833,—,4.54
2,Q03,장기주택저당 차임금 이자 공제한도,0.333,0.000,0.333,—,4.47
3,Q04,기부금이월공제기간몇년,0.333,0.796,0.000,—,2.68
4,Q05,중퇴자 연정 언제함?,0.429,0.764,1.000,—,5.34
5,Q06,월세공제 몇 퍼임?,0.111,0.000,0.333,—,3.75
6,Q07,dnjftp rhdwp 한도 알려줘,0.400,0.408,0.500,—,3.81
7,Q08,dusakfwjdtks 언제 해요?,0.222,0.404,1.000,—,5.10
8,Q09,Monthly rent 세액공제 rate가 몇 %?,0.400,0.000,1.000,—,2.73
9,Q10,외국인 flat tax 19% 고르면 deduction 못 받아요?,0.200,0.000,0.333,—,5.03


💵 이 셀의 GPT 사용 — RAGAS 심판(gpt-5-mini·none) 150회 · 입력 164,750 · 출력 25,731(추론 0) 토큰 → $0.0926  (저장된 점수 재사용분은 $0)
저장: artifacts/ragas_per_question.csv · artifacts/ragas_summary.csv


## 3. ✅ 최종 적용 파라미터 (01 학습 설정 + 02 생성·평가 설정)
> 👥 **담당:** PM · 통합

In [11]:
final = nb.final_params(CFG, extra={"01 학습 저장 시각": bi["saved_at"], "임베딩 모델(인덱스)": pipe.active.get("embed_model"),
                                    "프로필(02)": profile_name(HW), "CPU": HW.get("cpu_model"), "GPU": HW.get("gpu") or "없음",
                                    "평균 응답(s)": round(float(pd.Series([r.get("total_s") for r in answers], dtype=float).mean()), 2) if answers else None, "질문 수": len(EVAL_ITEMS), "평가 방식": EVAL_MODE, "답변 추론 강도(요청)": REASONING_EFFORT,
                                    "심판": f"{JUDGE_MODEL} / 추론 {JUDGE_REASONING_EFFORT}",
                                    "질문 해석": f"{REWRITE_MODEL} / 추론 {REWRITE_REASONING_EFFORT} / {REWRITE_N}개" if QUERY_REWRITE else "끔",
                                    "답변에 검색용 질문 반영": bool(QUERY_REWRITE and ANSWER_WITH_REWRITE),
                                    "리랭커": f"{RERANK_MODEL} (후보 {RERANK_CANDIDATES})" if RERANK else "끔", "심판 토큰(이번 실행)": dict(L.JUDGE_USAGE),
                                    "평가 종합(지표 평균)": ragas_summary.set_index("지표")["평균"].round(3).to_dict()},
                        save_to=str(Path(WORK_DIR) / "final_params.json"))

,구분,파라미터,값
0,로더(라이브러리),pdf_text_loader,PyPDFLoader
1,로더(라이브러리),pypdf,"{""mode"": ""page"", ""extraction_mode"": ""plain"", ""extraction_kwargs"": {}}"
2,로더(라이브러리),pdf_table_loader,PyMuPDFLoader
3,로더(라이브러리),table_settings,{}
4,로더(라이브러리),image_parser,hf-vlm
...,...,...,...
71,추가,질문 해석,gpt-5-mini / 추론 low / 2개
72,추가,답변에 검색용 질문 반영,True
73,추가,리랭커,끔
74,추가,심판 토큰(이번 실행),"{'calls': 150, 'prompt_tokens': 164750, 'completion_tokens': 25731, 'reasoning_tokens': 0}"


## 4. 💵 GPT API 비용 — 단계별 합계
> 👥 **담당:** PM · 통합
이번 실행에서 새로 호출한 비용과, 저장본(이전 실행)까지 포함한 질문 전체 비용을 함께 보여줍니다.

In [ ]:
# 4) 💵 GPT API 비용 — ① 질문 해석 · ④ 답변 생성 · RAGAS 심판 (이 셀은 GPT 를 호출하지 않음)
_rw_all = [x for x in (SR.rw_for(r["question"]) for r in answers) if x]
def _aug(r, k):
    return (r.get("augmentation") or {}).get(k) or 0
_rwN = SR.usage()
_ansN = globals().get("ANS_NEW") or dict(calls=0, prompt=0, completion=0, reasoning=0, usd=0)
_jN = globals().get("JUDGE_NEW") or dict(calls=0, prompt=0, completion=0, reasoning=0, usd=0)
_rw_tot = sum(x.get("usd") or 0 for x in _rw_all)
_ans_tot = usd(sum(_aug(r, "prompt_tokens") for r in answers), sum(_aug(r, "completion_tokens") for r in answers), LLM_MODEL) or 0
cost_df = pd.DataFrame([
    {"단계": "① 질문 해석", "모델 · 추론": f"{REWRITE_MODEL} · {REWRITE_REASONING_EFFORT}" if QUERY_REWRITE else "끔",
     "호출(이번)": _rwN["calls"], "입력 토큰(이번)": _rwN["prompt"], "출력 토큰(이번)": _rwN["completion"], "추론 토큰(이번)": _rwN["reasoning"],
     "비용(이번)": _rwN["usd"], "비용(질문 전체·저장본 포함)": _rw_tot},
    {"단계": "④ 답변 생성", "모델 · 추론": f"{LLM_MODEL} · {REASONING_EFFORT}",
     "호출(이번)": _ansN["calls"], "입력 토큰(이번)": _ansN["prompt"], "출력 토큰(이번)": _ansN["completion"], "추론 토큰(이번)": _ansN["reasoning"],
     "비용(이번)": _ansN["usd"], "비용(질문 전체·저장본 포함)": _ans_tot},
    {"단계": "RAGAS 심판", "모델 · 추론": f"{JUDGE_MODEL} · {JUDGE_REASONING_EFFORT}",
     "호출(이번)": _jN["calls"], "입력 토큰(이번)": _jN["prompt"], "출력 토큰(이번)": _jN["completion"], "추론 토큰(이번)": _jN["reasoning"],
     "비용(이번)": _jN["usd"], "비용(질문 전체·저장본 포함)": _jN["usd"]},
])
cost_df.loc[len(cost_df)] = ["합계", "", *cost_df[["호출(이번)", "입력 토큰(이번)", "출력 토큰(이번)", "추론 토큰(이번)", "비용(이번)", "비용(질문 전체·저장본 포함)"]].sum().tolist()]
display(cost_df.style.format("{:,.0f}", subset=["호출(이번)", "입력 토큰(이번)", "출력 토큰(이번)", "추론 토큰(이번)"])
        .format(fmt_usd, subset=["비용(이번)", "비용(질문 전체·저장본 포함)"]).hide(axis="index"))
print("※ ③ 리랭커·임베딩·BM25 는 내 GPU/CPU 에서 계산 → $0.  RAGAS 심판의 '전체'는 이번 실행분만 (저장된 점수의 예전 비용은 기록하지 않음)")

단계,모델 · 추론,호출(이번),입력 토큰(이번),출력 토큰(이번),추론 토큰(이번),비용(이번),비용(질문 전체·저장본 포함)
① 질문 해석,gpt-5-mini · low,10,"2,956","3,409","2,880",$0.007557,$0.009905
④ 답변 생성,gpt-5-mini · low,15,"26,970","6,887","4,096",$0.0205,$0.0205
RAGAS 심판,gpt-5-mini · none,150,"164,750","25,731",0,$0.0926,$0.0926
합계,,175,"194,676","36,027","6,976",$0.1207,$0.1231


※ ③ 리랭커·임베딩·BM25 는 내 GPU/CPU 에서 계산 → $0.  RAGAS 심판의 '전체'는 이번 실행분만 (저장된 점수의 예전 비용은 기록하지 않음)


: 